# Priminfo archive; Scraping Premium PDFs

We download the yearly premium PDFs from the archive of the Federal Office of Public Health (BAG)https://www.priminfo.admin.ch/de/downloads/archiv

Then we opened one pdf with **pdfplumber** to see how the tables inside look

The PDFs are our main source for the premium data. They are the official yearly premium overviews of the Federal Office of Public Health (BAG), published on priminfo.admin.ch
We collect them by web scraping (requests and BeautifulSoup) and read the premiums out of them with pdfplumber
`Praemien_CH.csv` from opendata.swiss serves as a cross-check

### Before start:
- In VS Code, click "Select Kernel" at the top right and choose the Python from `.venv`
   That is where `requests`, `beautifulsoup4` and `pdfplumber` are installed.
- The downloaded PDFs in `data/raw/` are stored in Git, so everyone in the team has the same files

## Problems we faced

While building this notebook we ran into several problems. The parser never drops lines silently:
it counts the lines it can't read and lists them per page, canton and model. That's how we found
each of these problems.

### Scraping

| Problem | Solution |
|---|---|
| The archive page lists many PDFs: 1996 until today, EU premiums (`praemien_eu_...`), premium subsidies (`praemienverbilligung_...`) | Keep only files named `praemien_YYYY.pdf`, for 2012–2022, because our income data (ESTV) ends with 2022 |
| Is the year in the file name the premium year? | Yes: `praemien_2012.pdf` was published in October 2011 and contains the premiums **for** 2012 |
| A User-Agent with contact details would show personal data | Team decision: no custom User-Agent. robots.txt is checked with `"*"` (the rules for every program). Pauses, caching and the request log stay in place |

### Reading the PDFs

| Problem | Solution |
|---|---|
| Over 1100 pages per PDF, page 1 is only the cover | Steps 5 and 6 explore the layout before parsing |
| No lines around the table cells, so `extract_tables()` finds 0 tables | Read the text line by line with regular expressions |
| Some insurers don't offer every deductible (e.g. no CHF 2000). In the text the gap disappears and numbers shift into the wrong column: **8,207** unreadable lines in 2012 | Match each premium to its deductible by its **position** on the page (`extract_words()`): down to **578** lines |
| The word "Région" also appears in other tables: `ValueError: 'No-OFSP'` | A line only counts as a header if a region **number** follows |
| Ticino (TI) pages are in Italian ("Regione", "Franchigia"). All 578 remaining problem lines were there, and TI was missing | Accept both languages: down to **0** problem lines in 2012 |
| Valais (VS) calls its models "Krankenpflege - Grundversicherung" etc., so VS was missing from the reference case | Remove "Krankenpflege - " from the model name |
| From 2013 on, year and canton ("2013 AG") sit inside the "Mit Unfall" line instead of on their own line. The code found **0 cantons** for 2013–2022 | Search for year + canton anywhere in the line, and only accept the 26 real canton codes |
| From 2019 on, the PDF uses a long dash ("Prämien – Grundversicherung") and no colon after "Franchise", so 2019–2022 were missing from the reference case | Turn "–" into "-", and make the colon optional (`:?`) |
| A small special section "Entsandte 20xx – Grundversicherung" (posted workers) appears from 2016 on | Not part of our reference case, so it is left out automatically |

### Result

- All 11 PDFs (2012–2022) read, with **26 cantons** each: **2,922,813** premium values
- Only 16 lines*(2016–2018) could not be read safely and were left out
- The canton file has **26 cantons for every year from 2012 to 2022**

# Preparation


In [24]:
# Imports 
import csv
import random #create random numbers; use it to wait a random time between requests to avoid being blocked
import time # working with time, e.g. to wait between requests
from datetime import datetime #timestamp for logging
from pathlib import Path # join  folder name corretly on Windows and macOS
from urllib.parse import urljoin, urlparse   # urljoin turns "/de/x.pdf" into a full web address: urlparse splits a URL into its components (scheme, domain, path, etc
from urllib.robotparser import RobotFileParser  # standard Python tool that reads robots.txt

import requests
from bs4 import BeautifulSoup # parse HTML
import pdfplumber #counts pages from 0

In [25]:
# Settings (!!change these here, not further down!!) 
# All values we might want to change are collected here in CAPITALS 

ARCHIVE_URL = "https://www.priminfo.admin.ch/de/downloads/archiv"

# Only PDFs for these premium years are downloaded and read
FIRST_YEAR = 2012
LAST_YEAR = 2022

# Pause between requests in seconds; random value in this range is used each time, so we do not hit the server in a fixed rhythm
MIN_WAIT = 2
MAX_WAIT = 5

# The notebook is in notebooks/, but the data folder is one level up in the project
# Path.cwd() folder the notebook is running in
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd() #makes the notebool work for every team member, no matter where they put the project folder on their computer

RAW_DIR = PROJECT_ROOT / "data" / "raw" #raw unchanged pdfs
PDF_DIR = RAW_DIR / "priminfo_pdfs" # downloades PDFs are saved here
PAGE_CACHE = RAW_DIR / "priminfo_archiv.html" #saved copy of archive page, so we do not have to download it every time we run the notebook
LOG_FILE = RAW_DIR / "download_log.csv" # log of all downloaded PDFs, with timestamp and HTTP status code

# Create the folders if they do not exist yet (parents=True also creates data/ and data/raw/)
PDF_DIR.mkdir(parents=True, exist_ok=True)
print("[setup] project root:", PROJECT_ROOT)
print("[setup] PDFs will be saved in:", PDF_DIR)

[setup] project root: c:\Users\bitsc\projects\CIP-project
[setup] PDFs will be saved in: c:\Users\bitsc\projects\CIP-project\data\raw\priminfo_pdfs


### Helper functions

Two functions that we use in every download

- `log_request` writes one line to  download_log.csv (time, URL, status code)
- `polite_get` downloads a URL with our User-Agent, logs it and then waits 2 to 5 seconds

Because every download goes through `polite_get`, we need the pause or the log

 HTTP status codes in short: `200` = OK ; `404` = page not found ; `403` = access denied ;`500` = error on the server

**Note**: this cell prints nothing. They run later when Steps 2 and 4 call them


In [26]:
# Helper functions 
def log_request(url, status):
    # does the log file already exists, if not write the header row first
    is_new = not LOG_FILE.exists()
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f: #opeens file in append mode, so we do not overwrite the log file each time we run the notebook; utf-8 makes special characters like "ü" work correctly;
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["timestamp", "url", "status"])
        writer.writerow([datetime.now().isoformat(timespec="seconds"), url, status])


def polite_get(url):
    response = requests.get(url, timeout=30) # Download the URL with a 30-second timeout (so notebook does not hang forever if the server is slow or down)
    log_request(url, response.status_code) # Log the request in the CSV file, so we can see which PDFs were downloaded successfully and which failed
    print(f"[download] {response.status_code}  {url}") # Print the download status and URL
    time.sleep(random.uniform(MIN_WAIT, MAX_WAIT)) 
    response.raise_for_status() #stop here with an error
    return response

## Step 1: Check robots.txt


In [27]:
host = urlparse(ARCHIVE_URL)            # splits the address into parts 
robots_url = f"{host.scheme}://{host.netloc}/robots.txt"

robots = RobotFileParser()
robots.set_url(robots_url)
robots.read()                           # downloads and reads robots.txt (a missing file counts as allow all)

allowed = robots.can_fetch("*", ARCHIVE_URL)   # "*" = the rules that apply to every program; not sen dour own user agent
print("[robots]", robots_url)
print("[robots] are we allowed to fetch the archive page?", allowed)

if not allowed:
    raise SystemExit("robots.txt forbids this page, so we stop here.")

[robots] https://www.priminfo.admin.ch/robots.txt
[robots] are we allowed to fetch the archive page? True


## Step 2: Download the archive page

**Caching:** from the second run on, the notebook reads the saved file and doesn't ask the server again (faster and kinder to the server)
If you want a fresh copy, delete `data/raw/priminfo_archiv.html` and run the cell again


In [ ]:
if PAGE_CACHE.exists():
    print("[page] using cached copy:", PAGE_CACHE) #we already downloaded the page before, so we do not have to download it again
    html = PAGE_CACHE.read_text(encoding="utf-8")
else:
    html = polite_get(ARCHIVE_URL).text #no copy yet, download page
    PAGE_CACHE.write_text(html, encoding="utf-8")
    print("[page] saved to:", PAGE_CACHE)

print("[page] length of the page in characters:", len(html))

[page] using cached copy: c:\Users\bitsc\projects\CIP-project\data\raw\priminfo_archiv.html
[page] length of the page in characters: 74603


## Step 3: Find all PDF links

`BeautifulSoup` goes through the page and looks at every link (`<a href="...">`). We keep the links whose address ends in `.pdf`

The archive page contains many links from 1996 until today and other kind of pdf, so here we keep only the links we neeed. (main premium overviews for 2012-2022)

In [28]:
import re   # regular expressions; way to search text for patterns

soup = BeautifulSoup(html, "html.parser")   # turns the HTML text into something we can search

pdf_links = []
for a in soup.find_all("a", href=True):     # every <a> tag that has an href attribute
    full_url = urljoin(ARCHIVE_URL, a["href"])
        # We only want the main premium overviews, named like "praemien_2012.pdf".
    # This skips praemien_eu (people living in the EU) and "praemienverbilligung" (premium subsidies), which are not part of our research questions
    file_name = Path(urlparse(full_url).path).name.lower()
    if not re.fullmatch(r"praemien_\d{4}\.pdf", file_name):   # fullmatch = the whole name must fit the pattern
        continue

    link_text = a.get_text(strip=True)      # the visible text of the link, e.g. "Prämien 2012"

    # Look for a year like 2012 first in the link text, then in the address.
    # \b means "word boundary", so 2012 is found but not the "2012" inside "120123"
    match = re.search(r"\b(20\d{2})\b", link_text) or re.search(r"(20\d{2})", full_url)
    if match is None:
        print("[links] no year found, skipped:", link_text, "->", full_url)
        continue

    year = int(match.group(1))              # group(1) is the year that was found, as a number
    if FIRST_YEAR <= year <= LAST_YEAR:
        pdf_links.append({"year": year, "url": full_url, "text": link_text})

# same file can appear twice on a page dict.fromkeys keeps the order and drops duplicates
unique_urls = list(dict.fromkeys(link["url"] for link in pdf_links))

print(f"[links] {len(pdf_links)} PDF links between {2012} and {2022}, {len(unique_urls)} unique")
for link in sorted(pdf_links, key=lambda l: l["year"]):   # sorted by year, so the list is easy to check
    print("   ", link["year"], "|", link["text"], "->", link["url"])

[links] no year found, skipped: Prämienübersicht 1999PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1999.pdf
[links] no year found, skipped: Prämienübersicht 1998PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1998.pdf
[links] no year found, skipped: Prämienübersicht 1997PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1997.pdf
[links] no year found, skipped: Prämienübersicht 1996PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1996.pdf
[links] 11 PDF links between 2012 and 2022, 11 unique
    2012 | Prämienübersicht 2012PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2012.pdf
    2013 | Prämienübersicht 2013PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2013.pdf
    2014 | Prämienübersicht 2014PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2014.pdf
    2015 | Prämienübersicht 2015PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2015.pdf
    2016 | Prä

## Step 4: Download the PDFs

Every PDF from Step 3 is saved in `data/raw/priminfo_pdfs/`
The file name is taken from theaddress.

Before downloading, we check whether the file already exists. If it does, it is skipped so we never download the same file twice


In [29]:
for url in unique_urls:
    file_name = Path(urlparse(url).path).name   # the last part of the address, e.g. "praemien_2012.pdf"
    target = PDF_DIR / file_name

    if target.exists():
        print("[skip] already downloaded:", file_name)
        continue

    response = polite_get(url)              #helper function
    target.write_bytes(response.content)        # .content is the raw file, write_bytes saves it unchanged
    print("[saved]", target)

print("[done] PDFs in folder:", len(list(PDF_DIR.glob("*.pdf"))))

[skip] already downloaded: praemien_2022.pdf
[skip] already downloaded: praemien_2021.pdf
[skip] already downloaded: praemien_2020.pdf
[skip] already downloaded: praemien_2019.pdf
[skip] already downloaded: praemien_2018.pdf
[skip] already downloaded: praemien_2017.pdf
[skip] already downloaded: praemien_2016.pdf
[skip] already downloaded: praemien_2015.pdf
[skip] already downloaded: praemien_2014.pdf
[skip] already downloaded: praemien_2013.pdf
[skip] already downloaded: praemien_2012.pdf
[done] PDFs in folder: 11


## Step 5: Look inside one PDF
We open one*PDF with pdfplumber and print:
1. the number of pages,
2. the plain text of page 1,
3. the tables pdfplumber finds on page 1 (the first 8 rows of each).

**Why don't parse everything yet**: Every PDF is built differently. The parsing code depends completely on the table layout; which columns there are, where the canton is, how the deductibles are listed. 
So we look first and write the parsing code afterwards

**How to look at another file**: Change `pdf_files[0]` to `pdf_files[1]`, `pdf_files[2]` and so on

In [30]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print("[pdf] files available:", [p.name for p in pdf_files])

# Take the first file. Change the 0 to look at another one.
example = pdf_files[0]
print("[pdf] opening:", example.name)

with pdfplumber.open(example) as pdf:            # "with" closes the file again automatically
    print("[pdf] number of pages:", len(pdf.pages))
    first_page = pdf.pages[0]

    print("\n----- plain text of page 1 (first 1500 characters) -----")
    print((first_page.extract_text() or "")[:1500])

    print("\n----- tables pdfplumber detects on page 1 -----")
    tables = first_page.extract_tables()          # a list of tables, each table is a list of rows
    print("number of tables:", len(tables))
    for i, table in enumerate(tables):
        print(f"\ntable {i}: {len(table)} rows")
        for row in table[:8]:                     # only the first 8 rows, to keep it readable
            print("   ", row)

[pdf] files available: ['praemien_2012.pdf', 'praemien_2013.pdf', 'praemien_2014.pdf', 'praemien_2015.pdf', 'praemien_2016.pdf', 'praemien_2017.pdf', 'praemien_2018.pdf', 'praemien_2019.pdf', 'praemien_2020.pdf', 'praemien_2021.pdf', 'praemien_2022.pdf']
[pdf] opening: praemien_2012.pdf
[pdf] number of pages: 1191

----- plain text of page 1 (first 1500 characters) -----
Eidgenössisches Departement des Innern EDI
Département fédéral de l'intérieur DFI CH
Bundesamt für Gesundheit BAG
Office fédéral de la santé publique OFSP
Oktober/octobre/ottobre 2011
Prämienübersicht
Aperçu des primes
Panoramica dei premi
2012
- Grundversicherungsprämien (Wahlfranchisen, HMO, Hausarzt und
weitere Modelle)
- Kündigungsfristen
- Sparmöglichkeiten
- Primes de l’assurance de base (franchises à option, HMO, médecin
de famille et autres modèles)
- Délais de résiliation
- Possibilités d’économies
- Premi dell’assicurazione di base (franchigie opzionale, HMO, medico
di famiglia e altri modelli)
- Termini di d

## Step 6: Find the pages with the premium tables
We look at a few pages spread through `praemien_2012.pdf`. For each page we print the text and the first rows of any table pdfplumber detects.

**Why**: PDF has over 1000 pages, and page 1 is only the cover . Before we can write the parsing code, we need to know where the premium tables start and how their layout is;
which columns there are, and where the insurer, canton, region, age group and deductible are.

In [31]:
# Look at a few pages further inside the PDF, to find where the premium tables start.
# Page numbers in pdfplumber start at 0, so pdf.pages[9] is page 10 of the PDF.
PAGES_TO_LOOK_AT = [2, 5, 10, 20, 50, 100]

example = PDF_DIR / "praemien_2012.pdf"

with pdfplumber.open(example) as pdf:
    for page_number in PAGES_TO_LOOK_AT:
        page = pdf.pages[page_number - 1]   # minus 1, because counting starts at 0
        text = page.extract_text() or ""
        tables = page.extract_tables()

        print("=" * 80)
        print(f"PAGE {page_number}   |   tables found: {len(tables)}")
        print("=" * 80)
        print(text[:800])                   # first 800 characters, so the output stays short

        if tables:
            print("\n--- first rows of the first table ---")
            for row in tables[0][:6]:
                print("   ", row)
        print()

PAGE 2   |   tables found: 0


PAGE 5   |   tables found: 0
Krankenversicherung: Information für die Versicherten
Das Krankenversicherungsgesetz (KVG) bietet Ihnen verschiedene Möglichkeiten, Ihre persönliche
Versicherungsprämie zu optimieren. Die vorliegende Information, vor allem das Kapitel „Sparmöglich-
keiten“, soll Ihnen dabei helfen und Ihnen zeigen, wie Sie Ihre Prämie spürbar senken können, ohne
Qualitätseinbussen beim Versicherungsschutz in Kauf nehmen zu müssen.
Auf dem Internet finden Sie zudem einen elektronischen Prämienvergleich, mit dem Sie die Prämien in
der Grundversicherung mit den verschiedenen Franchisen-Stufen der Krankenkassen in Ihrer Region
vergleichen können. Auch die Prämien bei HMO-, Hausarzt- (HAM) sowie weiteren Versicherungsmo-
dellen sind darin ersichtlich.
Die Internet-Adresse lautet: www.priminfo.ch.
Wünschen Sie w

PAGE 10   |   tables found: 0
Termini di disdetta
Cambiamento della cassa malati
- Avete la possibilità di stipulare l’assicurazione di ba

## Step 7: Read all premium rows out of one PDF
We go through every page of `praemien_2012.pdf` and read the premium rows line by line.
For each page we remember the model, the canton, the age group and the region from the page
header. Then each data row becomes 12 rows in our table -> 6 deductibles with accident and
6 deductibles without accident.

**Why read tex and not table:** pdfplumber found no tables (Step 5/6), because the PDF has no lines around its cells. The text,
however, is very regular, so we can read it with *regular expressions*  (patterns that describe what a line looks lik)

**Units** : The premiums in the PDF are **CHF per month**

**How a premium page is built (example):**
- Prämien - Grundversicherung (model)
- Mit Unfall / avec risque accidents 2013 AG ->year and canton (2012: on a line of its own)
- Franchise / Franchise: Erwachsene / Adultes ...  -> age group
- 300 500 1000 1500 2000 2500 Region / Région 0 300 500 ... 2500   -> deductibles + region
- 298.9 287.2 ... 179.3 Agrisano Brugg 284.0 272.8 ... 170.3  -> one insurer: 6 premiums WITH accident cover, the insurer, 6 premiums WITHOUT accident cover

**Functions in this step**
``pdfplumber`` gives every word with its position on the page:
    x0/x1 = left/right edge, top = distance from the top of the page
    Words with (almost) the same `top` belong to the same line
`group_into_lines(words)`
Puts the words of a page together into lines, from top to bottom.
pdfplumber gives every word with its position on the page (`x0`/`x1` = left/right edge,
`top` = distance from the top). Words with almost the same `top` belong to the same line

`middle(word)`
Returns the horizontal middle of a word on the page. We use it to compare positions.

`nearest_column(word, columns)`
Returns the deductible whose header is closest to a premium.
This is how we know that a premium belongs to e.g. CHF 2500 even when the insurer does not
offer CHF 2000 and that column is empty.

`parse_page(page, page_number, state, rows, problems)`
Reads all premium rows on one PDF page and adds them to `rows`.
- `state` remembers the header information (model, year, canton, age group, region,
  deductible positions). We keep it from page to page, because a table can continue on the
  next page without a new header 
- Rows we cannot read safely go to `problems` (with page, canton and model), so we see what
  we lose instead of dropping it silently


In [ ]:
import re
from collections import Counter
import pdfplumber
import pandas as pd

# Patterns: what the lines in the PDF look like
#r"..." is a "raw string": backslashes are kept as they are, which regular expressions need.
# ^ = start of the text, $ = end, \d = a digit, + = one or more, {4} = exactly 4 times.
NUMBER = re.compile(r"^\d+\.\d$")       # one premium, e.g. "180.2"
DEDUCTIBLE = re.compile(r"^\d+$")       # one deductible in the header, e.g. "300"
CANTON_PATTERN = re.compile(r"(?:^|\s)(20\d{2}) ([A-Z]{2})(?:\s|$)")   # "2012 AG" alone (2012) or inside a line (from 2013)
CANTONS = {"AG", "AI", "AR", "BE", "BL", "BS", "FR", "GE", "GL", "GR", "JU", "LU", "NE",
           "NW", "OW", "SG", "SH", "SO", "SZ", "TG", "TI", "UR", "VD", "VS", "ZG", "ZH"}
AGE_PATTERN = re.compile(r"Franchise / Franchi\w+:?\s*(.+?)\s*/")  # :? = colon or no colon (missing from 2019 on); Franchi\w+ matches "Franchise" and "Franchigia" (Ticino)# "Franchise:" (French) or "Franchigia:" (Italian, TI) Franchi\w+ means “Franchi” followed by any letters, so it matches both “Franchise” and “Franchigia”.
REGION_WORDS = ("Région", "Regione")    # the word after "Region /": French, or Italian for Ticino

LINE_TOLERANCE = 3    # words whose vertical position differs by less than this (in points) are on the same line


def group_into_lines(words): #put the words of a page together into lines, from top to bottom
   
    lines = []
    for word in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        if lines and abs(lines[-1][0]["top"] - word["top"]) < LINE_TOLERANCE:
            lines[-1].append(word)
        else:
            lines.append([word])
    return [sorted(line, key=lambda w: w["x0"]) for line in lines]   # each line from left to right


def middle(word):
    """Horizontal middle of a word on the page (in points)."""
    return (word["x0"] + word["x1"]) / 2


def nearest_column(word, columns): 
    #return the deductible whose header is closest to this premium and the distance
    # how we know that that a premium belongs to e.g. CHF 2500 even when the insurer does not offer CHF 2000 and that column is empty
    deductible, x = min(columns, key=lambda col: abs(col[1] - middle(word)))
    return deductible, abs(x - middle(word))


def parse_page(page, page_number, state, rows, problems):
   
    for line in group_into_lines(page.extract_words()):
        texts = [w["text"] for w in line]
        text = " ".join(texts)

        # Page title, e.g. "Prämien - HMO". "Prämien," and "Prämienübersicht" are explanation pages.
        if text.startswith("Prämien") and not text.startswith(("Prämien,", "Prämienübersicht")):
            # From 2019 on, the PDF uses a long dash "–" instead of "-": turn it into "-" first.
            model = text.replace("Prämien", "", 1).replace("–", "-").strip(" -")
            # Valais (VS) writes "Krankenpflege - Grundversicherung" etc. It is the same model,
            # so we remove "Krankenpflege - " to give all cantons the same model names.
            model = model.removeprefix("Krankenpflege - ")
            state["model"] = model or "(no model name)"
            continue

        # Year and canton: in 2012 a line of its own ("2012 AG"), from 2013 inside the "Mit Unfall" line.
        # We only accept real canton codes, and never on a data row (those start with a premium),
        # so that e.g. "2015 AG" inside an insurer name can't be mistaken for a canton.
        match = CANTON_PATTERN.search(text)
        if match and match.group(2) in CANTONS and not NUMBER.match(texts[0]):
            state["year"] = int(match.group(1))
            state["canton"] = match.group(2)
            continue

        # Age group line: "Franchise / Franchise: Erwachsene / Adultes ..." (Ticino: "Franchise / Franchigia: ...")
        if text.startswith("Franchise /"):
            match = AGE_PATTERN.search(text)
            if match:
                state["age_group"] = match.group(1)
            continue

        # Header with the deductibles: "300 500 ... Region / Région 0 300 500 ..."
        # Ticino writes "Regione" instead of "Région". We only accept it if a region NUMBER follows,
        # because other tables also contain the word (e.g. "Région No-OFSP").
        region_word = next((w for w in REGION_WORDS if w in texts), None)
        if region_word and texts.index(region_word) + 1 < len(texts) and texts[texts.index(region_word) + 1].isdigit():
            pos = texts.index(region_word)
            left_words, right_words = line[:pos], line[pos + 2:]   # pos+1 is the region number
            state["region"] = int(texts[pos + 1])
            # Remember each deductible AND where it stands on the page, so we can match premiums to it.
            state["cols_mit"] = [(int(w["text"]), middle(w)) for w in left_words if DEDUCTIBLE.match(w["text"])]
            state["cols_ohne"] = [(int(w["text"]), middle(w)) for w in right_words if DEDUCTIBLE.match(w["text"])]
            # A premium may be at most half a column away from its header, otherwise we don't trust the match.
            state["max_distance"] = min(b[1] - a[1] for a, b in zip(state["cols_mit"], state["cols_mit"][1:])) / 2
            continue

        # A data row starts and ends with a premium; everything in between is the insurer.
        if len(line) < 3 or not NUMBER.match(texts[0]) or not NUMBER.match(texts[-1]):
            continue
        n_left = 0
        while NUMBER.match(texts[n_left]):
            n_left += 1
        n_right = 0
        while NUMBER.match(texts[-1 - n_right]):
            n_right += 1
        insurer = " ".join(texts[n_left:len(texts) - n_right])

        if "cols_mit" not in state:
            problems.append((page_number, state.get("canton"), state.get("model"), "row before any deductible header", text))
            continue

        new_rows = []
        ok = True
        for accident, words, columns in [("mit_unfall", line[:n_left], state["cols_mit"]),
                                         ("ohne_unfall", line[len(line) - n_right:], state["cols_ohne"])]:
            used = set()
            for word in words:
                deductible, distance = nearest_column(word, columns)
                # Too far from every header, or two premiums for the same deductible: we do not trust it.
                if distance > state["max_distance"] or deductible in used:
                    ok = False
                used.add(deductible)
                new_rows.append({
                    "year": state.get("year"), "canton": state.get("canton"),
                    "region": state.get("region"), "age_group": state.get("age_group"),
                    "model": state.get("model"), "insurer": insurer,
                    "accident": accident, "deductible": deductible,
                    "premium_month": float(word["text"]), "page": page_number,
                })
        if ok:
            rows.extend(new_rows)
        else:
            problems.append((page_number, state.get("canton"), state.get("model"), "premium could not be matched to a deductible", text))


# Run it on the whole 2012 PDF
example = PDF_DIR / "praemien_2012.pdf"
rows, problems, state = [], [], {}

with pdfplumber.open(example) as pdf:
    for i, page in enumerate(pdf.pages, start=1):      # enumerate counts along: i = 1, 2, 3, ...
        parse_page(page, i, state, rows, problems)
        if i % 100 == 0:
            print(f"[parse] {i} of {len(pdf.pages)} pages done")

premiums = pd.DataFrame(rows)    # turn the list of rows into a table

print("\n[result] rows read:", len(premiums))
print("[result] problem lines:", len(problems))
print("\n[result] models:\n", premiums["model"].value_counts().to_string())
print("\n[result] age groups:\n", premiums["age_group"].value_counts().to_string())
print("\n[result] cantons found:", premiums["canton"].nunique(), sorted(premiums["canton"].dropna().unique()))

# Reference case per canton: do all 26 cantons have standard-model premiums for adults?
ref_cantons = premiums[(premiums["model"] == "Grundversicherung") & (premiums["age_group"] == "Erwachsene")]["canton"].unique()
print("\n[check] cantons with Grundversicherung for adults:", len(ref_cantons))
print("[check] missing:", sorted(set(premiums["canton"].dropna().unique()) - set(ref_cantons)) or "none")

# On which pages do the "Krankenpflege - ..." models appear? We look at them before deciding what they are.
kp = premiums[premiums["model"].str.startswith("Krankenpflege", na=False)]
print("\n[check] 'Krankenpflege' models on pages", kp["page"].min(), "to", kp["page"].max())

# Where are the problem lines? Counted per page, with canton and model, so we can open those pages.
# Counter counts how often each (page, canton, model) combination appears.
print("\n[check] problem lines per page:")
for (page_number, canton, model), count in sorted(Counter((p, c, m) for p, c, m, _, _ in problems).items()):
    print(f"   page {page_number} ({canton}, {model}): {count} lines")

print("\n[result] first 5 problem lines:")
for page_number, canton, model, reason, line in problems[:5]:
    print(f"   page {page_number} ({canton}, {model}): {reason}\n      {line}")


[parse] 100 of 1191 pages done
[parse] 200 of 1191 pages done
[parse] 300 of 1191 pages done
[parse] 400 of 1191 pages done
[parse] 500 of 1191 pages done
[parse] 600 of 1191 pages done
[parse] 700 of 1191 pages done
[parse] 800 of 1191 pages done
[parse] 900 of 1191 pages done
[parse] 1000 of 1191 pages done
[parse] 1100 of 1191 pages done

[result] rows read: 266843
[result] problem lines: 0

[result] models:
 model
Hausarztmodell                  112210
Grundversicherung                71483
Weitere Versicherungsmodelle     60186
HMO                              22964

[result] age groups:
 age_group
Kinder              95803
Erwachsene          85520
Junge Erwachsene    85520

[result] cantons found: 26 ['AG', 'AI', 'AR', 'BE', 'BL', 'BS', 'FR', 'GE', 'GL', 'GR', 'JU', 'LU', 'NE', 'NW', 'OW', 'SG', 'SH', 'SO', 'SZ', 'TG', 'TI', 'UR', 'VD', 'VS', 'ZG', 'ZH']

[check] cantons with Grundversicherung for adults: 26
[check] missing: none

[check] 'Krankenpflege' models on pages nan to n

In [33]:
# QUICK TEST: read only the first pages of every PDF, to check the code works for all years
# before starting the long Step 8. Takes about 1 minute instead of half an hour.
QUICK_PAGES = 60   # number of pages per PDF; the premium tables start around page 23

quick_rows = []
for pdf_path in sorted(PDF_DIR.glob("praemien_20??.pdf")):
    rows, problems, state = [], [], {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages[:QUICK_PAGES], start=1):   # [:60] = only the first 60 pages
            parse_page(page, i, state, rows, problems)
    quick_rows.extend(rows)
    print(f"[quick] {pdf_path.name}: {len(rows)} rows, {len(problems)} problem lines")

quick = pd.DataFrame(quick_rows)

# For every year: which models, age groups and cantons were found?
print("\n[check] per year:")
print(quick.groupby("year").agg(
    models=("model", lambda s: sorted(s.dropna().unique())),
    age_groups=("age_group", lambda s: sorted(s.dropna().unique())),
    cantons=("canton", lambda s: sorted(s.dropna().unique())),
).to_string())

# The most important check: does every year have rows for the reference case?
ref_quick = quick[(quick["model"] == "Grundversicherung") & (quick["age_group"] == "Erwachsene")]
print("\n[check] years with reference-case rows:", sorted(ref_quick["year"].unique()))

[quick] praemien_2012.pdf: 15348 rows, 0 problem lines
[quick] praemien_2013.pdf: 16306 rows, 0 problem lines
[quick] praemien_2014.pdf: 16184 rows, 0 problem lines
[quick] praemien_2015.pdf: 13816 rows, 0 problem lines
[quick] praemien_2016.pdf: 14254 rows, 0 problem lines
[quick] praemien_2017.pdf: 15000 rows, 0 problem lines
[quick] praemien_2018.pdf: 13532 rows, 0 problem lines
[quick] praemien_2019.pdf: 14072 rows, 0 problem lines
[quick] praemien_2020.pdf: 13956 rows, 0 problem lines
[quick] praemien_2021.pdf: 13588 rows, 0 problem lines
[quick] praemien_2022.pdf: 13634 rows, 0 problem lines

[check] per year:
                                                                      models                              age_groups       cantons
year                                                                                                                              
2012  [Grundversicherung, HMO, Hausarztmodell, Weitere Versicherungsmodelle]  [Erwachsene, Junge Erwachsene, Kinde

## Step 8: Read all PDFs (2012–2022) and save the result

We run the same `parse_page` function from Step 7 on every premium PDF, `praemien_2012.pdf`
to `praemien_2022.pdf`, and combine everything into one table. The table is saved as a CSV in
`data/processed/`.

**Why Step 7 used only 2012 in first attempt** Testing on one PDF first is much faster. Now that the code works for 2012, we use it for all years.

**Check the problem lines per year**
If one year has far more problem lines than the others, the PDF layout probably changed in that
year. Then we have to adjust the code for that year.

QUICK TEST: only the first and the last year. Delete this line to read all PDFs.
pdf_files = [pdf_files[0], pdf_files[-1]]

We run the same `parse_page` function from Step 7 on **all 11 PDFs** and combine everything
into one table. Step 7 must have run before, because it defines `parse_page`.

**functions used**

`pdf_files = sorted(PDF_DIR.glob("praemien_20??.pdf"))`Lists the main premium PDFs, sorted by year. `20??` means "20" followed by exactly two
characters, so `praemien_eu_...` and `praemienverbilligung_...` are skipped.

For each PDF:
- `state` starts empty, so no information (e.g. the last canton) carries over from the
  previous year.
- `parse_page` reads every page, exactly as in Step 7.
- The column `source_file` remembers which PDF each row comes from.
- A short summary is collected: rows read, problem lines, cantons.

`pd.concat(all_tables)`Glues the tables of all years together, one below the other

**Result:** all 11 PDFs (2012–2022) read with 26 cantons each; 2,922,813 premium values
Only 16 lines (2016–2018) could not be read safely and were left out


In [34]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_DIR / "priminfo_premiums_2012_2022.csv"

# Only the main premium PDFs, e.g. "praemien_2012.pdf". The glob pattern
# "praemien_20??.pdf" means: "praemien_20", then exactly 2 characters, then ".pdf".
# That skips "praemien_eu_..." and "praemienverbilligung_...".
pdf_files = sorted(PDF_DIR.glob("praemien_20??.pdf"))

print("[files]", [p.name for p in pdf_files])

all_tables = []       # one table per PDF, combined at the end
summary = []          # rows read and problems per year, for checking

for pdf_path in pdf_files:
    print(f"\n[parse] {pdf_path.name}")
    # Start with an empty state for every PDF, so nothing from the previous year carries over.
    rows, problems, state = [], [], {}

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            parse_page(page, i, state, rows, problems)
            if i % 200 == 0:
                print(f"   {i} of {len(pdf.pages)} pages done")

    table = pd.DataFrame(rows)
    table["source_file"] = pdf_path.name     # so we always know which PDF a row comes from
    all_tables.append(table)

    summary.append({
        "file": pdf_path.name,
        "rows_read": len(table),
        "problem_lines": len(problems),
        "cantons": table["canton"].nunique() if len(table) else 0,
    })
    print(f"   rows read: {len(table)}, problem lines: {len(problems)}")

# pd.concat glues the tables of all years together, one below the other.
premiums_all = pd.concat(all_tables, ignore_index=True)
premiums_all.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("\n[saved]", OUTPUT_FILE, "with", len(premiums_all), "rows")
print("\n[summary] per PDF:")
print(pd.DataFrame(summary).to_string(index=False))

[files] ['praemien_2012.pdf', 'praemien_2013.pdf', 'praemien_2014.pdf', 'praemien_2015.pdf', 'praemien_2016.pdf', 'praemien_2017.pdf', 'praemien_2018.pdf', 'praemien_2019.pdf', 'praemien_2020.pdf', 'praemien_2021.pdf', 'praemien_2022.pdf']

[parse] praemien_2012.pdf
   200 of 1191 pages done
   400 of 1191 pages done
   600 of 1191 pages done
   800 of 1191 pages done
   1000 of 1191 pages done
   rows read: 266843, problem lines: 0

[parse] praemien_2013.pdf
   200 of 1284 pages done
   400 of 1284 pages done
   600 of 1284 pages done
   800 of 1284 pages done
   1000 of 1284 pages done
   1200 of 1284 pages done
   rows read: 287446, problem lines: 0

[parse] praemien_2014.pdf
   200 of 1321 pages done
   400 of 1321 pages done
   600 of 1321 pages done
   800 of 1321 pages done
   1000 of 1321 pages done
   1200 of 1321 pages done
   rows read: 291422, problem lines: 0

[parse] praemien_2015.pdf
   200 of 1391 pages done
   400 of 1391 pages done
   600 of 1391 pages done
   800 of 

## Step 9: One premium per canton and year
Our income data is per canton and year, so the premium has to be too. Only then can we
calculate the burden (premium ÷ median income)

**Reference case^** So that premiums are comparable across cantons, we keep only:
| Setting | Value |
|---|---|
| Model | Grundversicherung (standard model) |
| Age group | Erwachsene (adults) |
| Deductible | CHF 300 (lowest) |
| Accident cover | with accident (`mit_unfall`) |

**Functions used****
1. **Read** the result of Step 8. Rows without year or canton are counted and removed
2. **Filter** to the reference case
3. **Average per region:** the mean over all insurers within each premium region
4. **Average per canton:** the mean over the regions of a canton
5. **Annual premium:** monthly premium × 12, because our burden uses the annual premium

`groupby` makes groups (e.g. all rows with the same year, canton and region),
and `.agg(...)` calculates one value per group

**Decisions**
- **Simple average over insurers:** we don't know how many people are insured with each
  insurer (market shares), so every insurer counts the same
- **Simple average over regions:** we don't have the population per premium region, so
  every region counts the same

**New columns**
| Column | Meaning |
|---|---|
| `premium_month` | average monthly premium, CHF |
| `premium_year` | `premium_month` × 12, CHF |
| `n_regions` | number of premium regions averaged in this canton |


In [35]:
# Read the result of Step 8 (so this step also works without re-running the 20-minute parsing).
premiums_all = pd.read_csv(OUTPUT_FILE)

# --- Reference case: so that premiums are comparable across cantons ------
REF_MODEL = "Grundversicherung"   # standard model
REF_AGE = "Erwachsene"            # adults
REF_DEDUCTIBLE = 300              # lowest deductible, CHF 300
REF_ACCIDENT = "mit_unfall"       # including accident cover

ref = premiums_all[
    (premiums_all["model"] == REF_MODEL)
    & (premiums_all["age_group"] == REF_AGE)
    & (premiums_all["deductible"] == REF_DEDUCTIBLE)
    & (premiums_all["accident"] == REF_ACCIDENT)
]
print("[filter] rows in the reference case:", len(ref))

# Step 1: average over insurers within each region.
# groupby = "make groups with the same year, canton and region", then .mean() per group.
per_region = (
    ref.groupby(["year", "canton", "region"], as_index=False)
       .agg(premium_month=("premium_month", "mean"),
            n_insurers=("insurer", "nunique"))   # how many insurers we averaged over
)

# Step 2: average over the regions of each canton.
per_canton = (
    per_region.groupby(["year", "canton"], as_index=False)
              .agg(premium_month=("premium_month", "mean"),
                   n_regions=("region", "nunique"))
)

# Our burden uses the ANNUAL premium, the PDF gives it per month.
per_canton["premium_year"] = (per_canton["premium_month"] * 12).round(2)
per_canton["premium_month"] = per_canton["premium_month"].round(2)

# --- Checks ----------------------------------------------------------------
print("\n[check] cantons per year (should be 26):")
print(per_canton.groupby("year")["canton"].nunique().to_string())

print("\n[check] highest and lowest premium per year:")
for year, group in per_canton.groupby("year"):
    high = group.loc[group["premium_month"].idxmax()]   # idxmax: the row with the highest value
    low = group.loc[group["premium_month"].idxmin()]
    print(f"   {year}: highest {high['canton']} {high['premium_month']} CHF/month, "
          f"lowest {low['canton']} {low['premium_month']} CHF/month")

CANTON_FILE = PROCESSED_DIR / "priminfo_premiums_per_canton.csv"
per_canton.to_csv(CANTON_FILE, index=False, encoding="utf-8")
print("\n[saved]", CANTON_FILE)

per_canton.head(26)


[filter] rows in the reference case: 19237

[check] cantons per year (should be 26):
year
2012    26
2013    26
2014    26
2015    26
2016    26
2017    26
2018    26
2019    26
2020    26
2021    26
2022    26

[check] highest and lowest premium per year:
   2012: highest BS 502.67 CHF/month, lowest AI 284.73 CHF/month
   2013: highest BS 514.55 CHF/month, lowest AI 293.64 CHF/month
   2014: highest BS 527.66 CHF/month, lowest AI 303.24 CHF/month
   2015: highest BS 549.43 CHF/month, lowest AI 316.13 CHF/month
   2016: highest GE 573.01 CHF/month, lowest AI 329.59 CHF/month
   2017: highest GE 596.33 CHF/month, lowest AI 351.22 CHF/month
   2018: highest GE 621.16 CHF/month, lowest AI 364.53 CHF/month
   2019: highest GE 641.67 CHF/month, lowest AI 376.0 CHF/month
   2020: highest GE 649.79 CHF/month, lowest AI 380.36 CHF/month
   2021: highest GE 653.36 CHF/month, lowest AI 382.06 CHF/month
   2022: highest GE 656.59 CHF/month, lowest AI 380.63 CHF/month

[saved] c:\Users\bitsc\proje

,year,canton,premium_month,n_regions,premium_year
0,2012,AG,347.60,1,4171.22
1,2012,AI,284.73,1,3416.80
2,2012,AR,302.54,1,3630.43
3,2012,BE,419.10,3,5029.17
4,2012,BL,391.30,2,4695.64
5,2012,BS,502.67,1,6032.03
6,2012,FR,366.60,2,4399.24
7,2012,GE,491.69,1,5900.31
8,2012,GL,325.44,1,3905.34
9,2012,GR,317.94,3,3815.31


## Step 9.1: Comparison without accident cover

ere we calculate the same canton premiums **without** accident cover (`ohne_unfall`), with exactly
the same method as in Step 9


In [37]:
# Step 9.1: Comparison without accident cover 
# Same method as Step 9, only REF_ACCIDENT_COMPARE = "ohne_unfall".
# Needs Step 9 to have run before (it uses premiums_all, the other REF_ settings and per_canton)

REF_ACCIDENT_COMPARE = "ohne_unfall"   # without accident cover

# Filter: standard model, adults, CHF 300, but WITHOUT accident cover
ref_ohne = premiums_all[
    (premiums_all["model"] == REF_MODEL)
    & (premiums_all["age_group"] == REF_AGE)
    & (premiums_all["deductible"] == REF_DEDUCTIBLE)
    & (premiums_all["accident"] == REF_ACCIDENT_COMPARE)
]
print("[filter] rows without accident cover:", len(ref_ohne))

# Average 1: over insurers within each region (exactly as in Step 9).
per_region_ohne = (
    ref_ohne.groupby(["year", "canton", "region"], as_index=False)
            .agg(premium_month=("premium_month", "mean"))
)

# Average 2: over the regions of each canton.
per_canton_ohne = (
    per_region_ohne.groupby(["year", "canton"], as_index=False)
                   .agg(premium_month=("premium_month", "mean"),
                        n_regions=("region", "nunique"))
)
per_canton_ohne["premium_year"] = (per_canton_ohne["premium_month"] * 12).round(2)
per_canton_ohne["premium_month"] = per_canton_ohne["premium_month"].round(2)

# Check: 26 cantons per year?
print("\n[check] cantons per year (should be 26):")
print(per_canton_ohne.groupby("year")["canton"].nunique().to_string())

# Save under a DIFFERENT name, so the main file from Step 9 is not overwritten.
CANTON_FILE_OHNE = PROCESSED_DIR / "priminfo_premiums_per_canton_ohne_unfall.csv"
per_canton_ohne.to_csv(CANTON_FILE_OHNE, index=False, encoding="utf-8")
print("\n[saved]", CANTON_FILE_OHNE)

# Comparison: with vs. without accident cover 
# merge puts both tables side by side, matched by year and canton (like a join in Excel)
# suffixes adds "_mit" and "_ohne" to the column names, so we can tell them apart
compare = per_canton.merge(per_canton_ohne, on=["year", "canton"], suffixes=("_mit", "_ohne"))

# How much cheaper is "without accident cover", in percent?
compare["difference_pct"] = ((compare["premium_month_ohne"] / compare["premium_month_mit"] - 1) * 100).round(1)

print("\n[compare] same ranking of cantons? (1.0 = identical order)")
for year, group in compare.groupby("year"):
    # rank() turns premiums into places (1 = cheapest ... 26 = most expensive).
    # corr() then compares the two rankings: 1.0 = exactly the same order.
    # (This is the Spearman correlation, calculated without the extra library scipy.)
    same_order = group["premium_month_mit"].rank().corr(group["premium_month_ohne"].rank())
    print(f"   {year}: {same_order:.3f}")

compare[["year", "canton", "premium_month_mit", "premium_month_ohne", "difference_pct"]].head(26)

[filter] rows without accident cover: 19237

[check] cantons per year (should be 26):
year
2012    26
2013    26
2014    26
2015    26
2016    26
2017    26
2018    26
2019    26
2020    26
2021    26
2022    26

[saved] c:\Users\bitsc\projects\CIP-project\data\processed\priminfo_premiums_per_canton_ohne_unfall.csv

[compare] same ranking of cantons? (1.0 = identical order)
   2012: 1.000
   2013: 1.000
   2014: 0.999
   2015: 0.999
   2016: 1.000
   2017: 0.999
   2018: 0.999
   2019: 1.000
   2020: 1.000
   2021: 0.999
   2022: 0.999


,year,canton,premium_month_mit,premium_month_ohne,difference_pct
0,2012,AG,347.60,326.38,-6.1
1,2012,AI,284.73,267.12,-6.2
2,2012,AR,302.54,283.82,-6.2
3,2012,BE,419.10,393.61,-6.1
4,2012,BL,391.30,367.22,-6.2
5,2012,BS,502.67,471.72,-6.2
6,2012,FR,366.60,343.90,-6.2
7,2012,GE,491.69,461.20,-6.2
8,2012,GL,325.44,305.37,-6.2
9,2012,GR,317.94,298.83,-6.0
